In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

# train stays on cpu (unfold is a view, moving it would copy every window), batches move to device
x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len).to(device=device)

In [4]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.input_linear = nn.Linear(in_features=embedding_size, out_features=hidden_size)
        self.hidden_linear = nn.Linear(in_features=hidden_size, out_features=hidden_size)
        self.output_linear = nn.Linear(in_features=hidden_size, out_features=vocab_size)

    def forward(self, input, h=None):
        batch_size = input.size(dim=0)
        if h is None:
            h = torch.zeros(size=(batch_size, self.hidden_size), device=input.device)
        logits = []
        for i in range(input.size(dim=1)):
            ch = input[:, i]
            new_h = torch.tanh(input=self.input_linear(input=self.embedding(input=ch)) + self.hidden_linear(input=h))
            logits.append(self.output_linear(input=new_h))
            h = new_h
        return torch.stack(tensors=logits, dim=1), h

In [5]:
# model initialization
model = RecurrentNeuralNetwork(vocab_size=vocab_size, embedding_size=embedding_size, hidden_size=hidden_size).to(device=device)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 93025


In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch].to(device=device)
    y_batch = y_train[batch].to(device=device)
    train_logits, _ = model(input=x_batch)
    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_batch.view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits, _ = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.209389686584473, val loss = 4.212765693664551
Step 1000: train loss = 1.5857326984405518, val loss = 1.7054929733276367
Step 2000: train loss = 1.467234492301941, val loss = 1.6575884819030762
Step 3000: train loss = 1.4579392671585083, val loss = 1.6463433504104614
Step 4000: train loss = 1.461545467376709, val loss = 1.6343315839767456
Step 4999: train loss = 1.3962490558624268, val loss = 1.6389062404632568


In [7]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long, device=device)

    while len(output) < max_output_length:
        logits, h = model(input=i, h=h)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long, device=device)

In [8]:
print(''.join(output))

ROMEO:
It is,
Rance the subject friend,
That dark'd friend likep!
So thou do.
Howast,
His comore men a sigred, your tongue for coward,
Froth friends in Anttany,
And be was then; scerenges think it in a harm home, and, you mind shall havry death.
Rimes, now nor horses upon the letters had yet thyself kind
helse, good friend my happy birthsissity?

ANGELO:
That Pleys, and yet me it
With fid then to they and is ere in my sakly,
What is to glid me about irny power is so, that thou wast false:
Who in't you'an advise her,
As he had! bed of my as
Aers
From a friend
When sawch my kneed!

FROTH:
Ayw to Yorkneyed
Hath here you person.

WALYON:
No friends and tooking such love my
by a crown?

DUKE VINCENTIO:

GLOUCESTER:
By headed but boy.
I'll gnoral,
Is lowk not, and stay
From us differenuing as to you in the gods, still change were this regring;
I was Coriolanus,
Emtant days?

Mescauth thou deny, and
Is sake.
Here's not with so crown is an eat die all than our lovely long,
Will thou'lature
Tha